# AI651 PA3: Graph neural networks for traffic forecasting
Runtime: Colab GPU (T4). A CPU run works with one seed but takes hours.

## Setup
Install packages, get the code and the LargeST files. Kaggle keys come from Colab secrets KAGGLE_USERNAME and KAGGLE_KEY.

In [1]:
REPO_URL = "https://github.com/BR-20/PA3_DL4STG_BR.git"
STUDENT_NAME, ROLL_NUMBER = "Bimal Rehman", "25280081"
import os, sys
if "google.colab" in sys.modules:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/PA3/V3"
    os.makedirs(WORK, exist_ok=True)
    if not os.path.exists(f"{WORK}/repo"):
        !git clone -q {REPO_URL} {WORK}/repo
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    os.environ.pop("KAGGLE_API_TOKEN", None)
    %cd {WORK}/repo
!pip install -q tables pypandoc_binary
!pip install -q -U kaggle

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/PA3/V3/repo


In [2]:
DATA_DIR = "/content/data" if "google.colab" in sys.modules else "data"
FILES = ["ca_meta.csv", "ca_rn_adj.npy", "ca_his_raw_2020.h5", "ca_his_raw_2021.h5"]
os.makedirs(DATA_DIR, exist_ok=True)
for name in FILES:
    if not os.path.exists(f"{DATA_DIR}/{name}"):
        !kaggle datasets download -d liuxu77/largest -f {name} -p {DATA_DIR} -q
        if os.path.exists(f"{DATA_DIR}/{name}.zip"):
            !unzip -q -o {DATA_DIR}/{name}.zip -d {DATA_DIR} && rm {DATA_DIR}/{name}.zip
    print(name, round(os.path.getsize(f"{DATA_DIR}/{name}") / 1e6, 1), "MB")

ca_meta.csv 0.6 MB
ca_rn_adj.npy 591.7 MB
ca_his_raw_2020.h5 7253.1 MB
ca_his_raw_2021.h5 7233.3 MB


In [3]:
import json
import numpy as np
import pandas as pd
import torch
import pa3_toy as toy
import pa3_data as dt
import pa3_pipeline as pl
import pa3_analysis as an
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEEDS = [0, 1, 2] if DEVICE == "cuda" else [0]
CFG = {"d": 32, "d_e": 10, "alpha": 0.5, "lr": 1e-3, "batch_size": 64, "max_epochs": 30, "patience": 5, "top_k_analysis": 10}
FIG, RES, CKPT = "figures", "results", "checkpoints"
for folder in (FIG, RES, CKPT):
    os.makedirs(folder, exist_ok=True)
HARDWARE = pl.hardware_string(DEVICE)
print(HARDWARE, "seeds", SEEDS)

{'device': 'cuda', 'gpu': 'Tesla T4', 'cpu': 'Intel(R) Xeon(R) CPU @ 2.00GHz', 'cpu_threads': 2, 'torch': '2.14.1+cu130', 'python': '3.13.16'} seeds [0, 1, 2]


## Question 1
Exhaustive checks of the hand derivations.

In [4]:
TOY = {"q1a": toy.q1a_check(), "q1b": toy.q1b_check(K=8), "q1c": toy.q1c_check(max_size=8)}
print(TOY["q1a"])
print(TOY["q1b"])
pd.DataFrame([{"Task": r["task"], "Method": r["method"], "Possible": r["possible"], "Witness": r["witness"]} for r in TOY["q1c"]["rows"]])

{'hood_a': [-1, 1], 'hood_b': [0, 0], 'n_v': 2, 'S_v': 0, 'steady_a': 0, 'steady_b': 2, 'trials': 100}
{'K': 8, 'phi': {'-1': 1, '0': 9, '1': 81}, 'n_triples': 164, 'n_distinct_sums': 164, 'max_sum': 648, 'base_K_collision_sum': 8, 'nonsteady_checked_up_to': 50}


,Task,Method,Possible,Witness
0,(i) Count steady neighbours,Sum,True,None
1,(i) Count steady neighbours,Mean,False,"{'hood_1': (0, 1, 0), 'hood_2': (0, 2, 0), 've..."
2,(i) Count steady neighbours,Max,False,"{'hood_1': (0, 1, 0), 'hood_2': (0, 2, 0), 've..."
3,(ii) More than half slowing,Sum,True,None
4,(ii) More than half slowing,Mean,True,None
5,(ii) More than half slowing,Max,False,"{'hood_1': (1, 0, 1), 'hood_2': (2, 0, 1), 've..."
6,(iii) Any neighbour slowing,Sum,True,None
7,(iii) Any neighbour slowing,Mean,True,None
8,(iii) Any neighbour slowing,Max,True,None


## Question 2
Ring road fits, rollout heatmaps and exact rules.

In [5]:
TOY["q2a_fit"] = toy.q2a_fit()
TOY["q2a_rollout"] = toy.q2a_rollout(TOY["q2a_fit"]["a"], TOY["q2a_fit"]["b"], fig_path=f"{FIG}/q2a_heatmaps.png")
TOY["q2b"] = toy.q2b_check()
TOY["q2c"] = toy.q2c_check()
pl.save_json(TOY, f"{RES}/toy_results.json")
pd.Series({f"{part}.{k}": v for part in ("q2a_fit", "q2a_rollout", "q2b", "q2c") for k, v in TOY[part].items() if not isinstance(v, list)}).round(4)

,0
q2a_fit.a,0.0000
q2a_fit.b,0.5000
q2a_fit.sse,0.5000
q2a_rollout.pred_peak_sensor_t20,20.0000
q2a_rollout.pred_peak_height_t20,0.4047
q2a_rollout.true_peak_sensor_t20,0.0000
q2a_rollout.true_peak_height_t20,1.0000
q2a_rollout.width_t0,5.0000
q2a_rollout.pred_width_t20,11.0000
q2a_rollout.true_width_t20,5.0000


## Q3(a) Data extraction and resampling
5 minute counts are summed to 15 minute counts.

In [6]:
SD_META, FLOW, Q3A = pl.q3a(DATA_DIR, FIG)
print({k: v for k, v in Q3A.items() if k not in ("missing", "resample_checks", "week_plot")})
print(pd.DataFrame(Q3A["resample_checks"]))
pd.Series(Q3A["missing"]).round(3)

{'n_sensors': 716, 'first_stamp': '2020-01-01 00:00:00', 'last_stamp': '2021-12-31 23:45:00', 'T_15min': 70176, 'T_5min': 210528, 'raw_integer_share': 1.0, 'week_sensors': ['1114720', '1117835', '1117717']}
                  time   sensor        five_min_values  fifteen_min_value
0  2021-04-07 22:15:00  1111547  [223.0, 205.0, 234.0]              662.0
1  2021-01-04 23:15:00  1121724     [55.0, 57.0, 64.0]              176.0
2  2021-09-12 16:30:00  1108599  [309.0, 326.0, 284.0]              919.0


,0
raw_5min_cells,1.507380e+08
raw_5min_missing,7.636666e+06
raw_5min_missing_pct,5.066000e+00
raw_rows_added_by_reindex,0.000000e+00
flow_15min_cells,5.024602e+07
flow_15min_missing,2.550289e+06
flow_15min_missing_pct,5.076000e+00
sensors_with_any_missing,7.160000e+02
sensors_over_50pct_missing,4.300000e+01
max_sensor_missing_pct,1.000000e+02


## Q3(b) Constructing the traffic graph

In [7]:
OFF, OUT_DEG, Q3B = pl.q3b(DATA_DIR, SD_META, FLOW, FIG)
print(Q3B["ordering"])
pd.Series(Q3B["stats"]).round(4)

{'first_three': [('1114091', 6931), ('1118333', 6932), ('1118348', 6933)], 'n': 716}


,0
nodes,716
nonzero_edges_offdiag,17319
diag_nonzero,716
symmetric,False
max_abs_asym,1.0
asym_ordered_pairs,21022
asym_unordered_pairs,10511
density_pct,3.383014
weight_min,0.010002
weight_max,1.0


## Q3(c) Graph neighbourhoods

In [8]:
Q3C = pl.q3c(OFF, OUT_DEG, SD_META)
{k: v for k, v in Q3C.items()}

{'node_index': 256,
 'node_id': '1108358',
 'one_hop_count': 20,
 'two_hop_count': 48,
 'one_hop_ids': ['1111530',
  '1122637',
  '1122636',
  '1122645',
  '1122646',
  '1115620',
  '1115477',
  '1116593',
  '1116594',
  '1108385',
  '1108356',
  '1111561',
  '1108387',
  '1108353',
  '1111563',
  '1115612',
  '1115537',
  '1115616',
  '1115541',
  '1111575'],
 'two_hop_only_ids': ['1111535',
  '1115463',
  '1111532',
  '1111531',
  '1115624',
  '1115486',
  '1115490',
  '1115628',
  '1127026',
  '1118767',
  '1108341',
  '1108423',
  '1108347',
  '1115608',
  '1108351',
  '1111569',
  '1108389',
  '1108419',
  '1113364',
  '1111555',
  '1111557',
  '1113915',
  '1113222',
  '1113894',
  '1108326',
  '1111554',
  '1111556',
  '1113904']}

## Q4(a) Forecasting windows
Windows stay inside one split. Normalisation uses 2020 only.

In [9]:
SERIES, Q4A = pl.q4a(FLOW)
print(Q4A["norm"], Q4A["window_counts"], "normalisation round trip max diff", Q4A["norm_roundtrip_max_diff"])
pd.DataFrame(Q4A["window_checks"]).T

{'mean': 607.16998318722, 'std': 483.3516052269983, 'train_last_stamp': '2020-12-31 23:45:00', 'n_values': 24122721} {'train': 35125, 'val': 8629, 'test': 26389} normalisation round trip max diff 4.547473508864641e-13


,t,input_stamps,target_stamps,x_shape,y_shape,first_window_start,last_window_end
train,7,"[2020-01-01 00:00:00, 2020-01-01 01:45:00]","[2020-01-01 02:00:00, 2020-01-01 02:45:00]","[8, 716, 1]","[4, 716, 1]",2020-01-01 00:00:00,2020-12-31 23:45:00
val,7,"[2021-01-01 00:00:00, 2021-01-01 01:45:00]","[2021-01-01 02:00:00, 2021-01-01 02:45:00]","[8, 716, 1]","[4, 716, 1]",2021-01-01 00:00:00,2021-03-31 23:45:00
test,7,"[2021-04-01 00:00:00, 2021-04-01 01:45:00]","[2021-04-01 02:00:00, 2021-04-01 02:45:00]","[8, 716, 1]","[4, 716, 1]",2021-04-01 00:00:00,2021-12-31 23:45:00


## Q4(b) Model checks
Shapes, row sums and the graph layer against a plain loop.

In [10]:
FIXED_NORM, MODEL_CHECKS = pl.model_checks(OFF, dt.N_NODES, DEVICE, CFG, SERIES)
pd.DataFrame(MODEL_CHECKS).T

,z_shape,y_shape,n_params,graph_loop_max_diff
none,"[4, 716, 32]","[4, 4, 716]",8516,NaN
fixed,"[4, 716, 32]","[4, 4, 716]",8516,0.0
adaptive,"[4, 716, 32]","[4, 4, 716]",22836,0.0
hybrid,"[4, 716, 32]","[4, 4, 716]",22836,0.0


## Q4(b), Q4(c), Q5 Training
Temporal only, fixed, adaptive and hybrid models with the same protocol.

In [11]:
RUNS, MODELS = pl.run_experiments(SERIES, FIXED_NORM, dt.N_NODES, SEEDS, DEVICE, CFG, CKPT)
pl.save_json(RUNS, f"{RES}/runs.json")
TABLE = pl.summary_table(RUNS)
TABLE.to_csv(f"{RES}/summary.csv", index=False)
PROTOCOL = pl.protocol_table(RUNS, SERIES)
print(PROTOCOL.to_string())
TABLE.round(2)

none 0 {'mae': 65.43, 'rmse': 102.62} time_s 407.0 epochs 20
none 1 {'mae': 64.93, 'rmse': 102.17} time_s 593.3 epochs 30
none 2 {'mae': 65.13, 'rmse': 102.34} time_s 532.9 epochs 27
fixed 0 {'mae': 144.9, 'rmse': 221.62} time_s 614.2 epochs 30
fixed 1 {'mae': 143.14, 'rmse': 220.21} time_s 613.2 epochs 30
fixed 2 {'mae': 142.43, 'rmse': 219.6} time_s 647.6 epochs 30
adaptive 0 {'mae': 99.63, 'rmse': 149.0} time_s 564.6 epochs 27
adaptive 1 {'mae': 101.82, 'rmse': 154.09} time_s 644.1 epochs 30
adaptive 2 {'mae': 89.97, 'rmse': 137.16} time_s 624.8 epochs 30
hybrid 0 {'mae': 87.47, 'rmse': 132.84} time_s 640.2 epochs 30
hybrid 1 {'mae': 88.72, 'rmse': 134.59} time_s 632.3 epochs 30
hybrid 2 {'mae': 86.81, 'rmse': 131.08} time_s 681.4 epochs 30
              seeds   d  d_e  alpha     lr  batch_size  max_epochs  patience  train_windows  val_windows  test_windows
graph                                                                                                                 
none    

,Model,Adjacency,graph,n_seeds,val_mae_mean,val_mae_std,val_rmse_mean,val_rmse_std,test_mae_mean,test_mae_std,test_rmse_mean,test_rmse_std,train_s_mean,train_s_std,epochs_mean,capped_runs,best_epoch_mean,s_per_epoch_mean,final_train_loss_mean
0,Temporal-only,None,none,3,57.09,0.39,91.58,0.42,65.17,0.25,102.38,0.22,511.07,95.05,25.67,1,21.33,19.95,0.11
1,STGNN,Fixed,fixed,3,127.48,1.06,202.15,0.80,143.49,1.27,220.48,1.03,624.98,19.60,30.00,3,29.67,20.83,0.26
2,STGNN,Adaptive,adaptive,3,82.18,7.08,131.49,10.61,97.14,6.30,146.75,8.68,611.13,41.47,29.00,2,25.67,21.07,0.13
3,STGNN,Hybrid,hybrid,3,74.93,0.88,120.00,1.78,87.67,0.97,132.83,1.75,651.33,26.36,30.00,3,27.67,21.71,0.13


## Q5(b), Q6(b) Learned graph analysis

In [12]:
GRAPH, FIXED_EFF = pl.q5_q6_graph_analysis(MODELS, SEEDS, FIXED_NORM, OFF, SD_META, FLOW, FIG, CKPT, DEVICE, CFG)
pl.save_json(GRAPH, f"{RES}/graph_analysis.json")
pd.DataFrame(GRAPH).drop(columns="strongest_absent_example").round(3)

,k,n_edges,share_in_road_pct,chance_in_road_pct,share_in_road_either_pct,chance_in_road_either_pct,median_km_learned,median_km_road,median_km_random,absent_from_road_count,absent_from_road_pct,corr_learned_absent,corr_random,max_row_weight_mean,uniform_weight,effective_neighbours
0,10,7160,5.936,3.383,7.360,4.108,21.264,2.371,23.529,6633,92.640,0.871,0.844,0.341,0.001,7.179
1,10,7160,6.494,3.383,7.919,4.108,20.256,2.371,23.529,6593,92.081,0.868,0.844,0.328,0.001,7.356
2,10,7160,6.592,3.383,8.017,4.108,20.152,2.371,23.529,6586,91.983,0.867,0.844,0.304,0.001,8.034


## Q4(c), Q6(b) Graph diagnostics
Self weight and how far each graph pulls a sensor away from its own training level.

In [13]:
DIAG = pl.graph_diagnostics(FLOW, FIXED_NORM, SEEDS, CKPT, DEVICE, CFG)
pl.save_json(DIAG, f"{RES}/diagnostics.json")
pd.DataFrame({g: DIAG[g] for g in ("fixed", "adaptive", "hybrid")}).round(3)

,fixed,adaptive,hybrid
self_weight_mean,0.173,0.029,0.096
effective_neighbours,13.725,7.523,14.089
level_gap_mean,97.289,73.551,79.906
level_gap_pct_of_mean_level,16.094,12.167,13.218


## Q6(a) Predicted against actual

In [14]:
BEST = min(["fixed", "adaptive", "hybrid"], key=lambda g: TABLE.set_index("graph").loc[g, "val_mae_mean"])
week_cols = [list(FLOW.columns).index(c) for c in Q3A["week_sensors"]]
PLOT_MODELS = {"Temporal-only": MODELS["none"], f"STGNN {an.LABELS[BEST][1]}": MODELS[BEST]}
PRED = an.plot_pred_vs_actual(PLOT_MODELS, SERIES["test"], week_cols, Q3A["week_sensors"], pl.WEEK_TEST, DEVICE, f"{FIG}/q6a_pred_vs_actual.png")
PRED["best_graph"] = BEST
pl.save_json(PRED, f"{RES}/pred_plot.json")
PRED

{'start': '2021-04-05 00:00:00',
 'end': '2021-04-11 23:45:00',
 'step_ahead': 4,
 'week_mae': {'Temporal-only': 75.32683563232422,
  'STGNN Hybrid': 73.24720001220703},
 'best_graph': 'hybrid'}

## Q7 Save results, versions and report

In [15]:
pl.save_json({"q3a": Q3A, "q3b": Q3B, "q3c": Q3C, "q4a": Q4A, "model_checks": MODEL_CHECKS, "cfg": CFG, "seeds": SEEDS,
              "hardware": HARDWARE, "fixed_effective_neighbours": FIXED_EFF}, f"{RES}/data_results.json")
!pip freeze | grep -iE "^(torch|numpy|pandas|h5py|tables|matplotlib|pypandoc_binary|kaggle)==" | sed 's/+cu[0-9]*//' > requirements.txt
!cat requirements.txt
REPO_LINK = REPO_URL[:-4] if REPO_URL.endswith(".git") else REPO_URL
!python build_report.py --name "{STUDENT_NAME}" --roll "{ROLL_NUMBER}" --repo "{REPO_LINK}" --out report.docx
for path in sorted(os.listdir(FIG)) + sorted(os.listdir(RES)):
    print(path)

h5py==3.14.0
kaggle==2.2.5
matplotlib==3.10.0
numpy==2.4.6
pandas==2.2.3
pypandoc_binary==1.17
tables==3.10.2
torch==2.14.1
wrote report.docx
q2a_heatmaps.png
q3a_week.png
q3b_adjacency.png
q3b_degree.png
q5b_learned_adjacency.png
q6a_pred_vs_actual.png
data_results.json
diagnostics.json
graph_analysis.json
pred_plot.json
previous_summary.csv
runs.json
summary.csv
toy_results.json
